# TrOCR Apex Killfeed — Fine-tuning on Colab GPU

**Setup steps before running:**
1. Upload a zip to Google Drive containing:
   - `labels/labels_clean.csv`
   - `crops/` directory
   - `crops_noise/` directory (if it exists)
   - `models/trocr_apex/` directory (your current checkpoint)
2. Set `DRIVE_ZIP_PATH` in Cell 2 to match where you uploaded it.
3. Runtime → Change runtime type → **T4 GPU**
4. Run all cells top to bottom.

The trained model will be saved back to Google Drive at the end.

In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — Config: set these to match your Drive layout
DRIVE_ZIP_PATH = "/content/drive/MyDrive/trocr_apex_data.zip"  # path to your uploaded zip
WORK_DIR       = "/content/trocr_apex"                          # where we extract to
MODEL_OUTPUT   = f"{WORK_DIR}/models/trocr_apex"                # saved back here each best epoch
DRIVE_MODEL_OUT = "/content/drive/MyDrive/trocr_apex_model"    # final model copied here

EPOCHS     = 10
BATCH_SIZE = 16   # T4 can handle 16; drop to 8 if you get OOM
LR         = 5e-5
QUALITY    = {"high", "medium", "noise"}

In [ ]:
# Cell 3 — Install dependencies & extract data
!pip install -q transformers==4.40.0 safetensors

import os, shutil, zipfile

os.makedirs(WORK_DIR, exist_ok=True)
print("Extracting zip...")
with zipfile.ZipFile(DRIVE_ZIP_PATH, "r") as z:
    z.extractall(WORK_DIR)
print("Done. Contents:")
!ls {WORK_DIR}

In [ ]:
# Cell 4 — Verify GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
# Cell 5 — Dataset + helpers (copied from train_trocr.py)
import csv, random, math, shutil, tempfile
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, get_linear_schedule_with_warmup

LABELS_CSV = Path(WORK_DIR) / "labels" / "labels_clean.csv"

class KillfeedDataset(Dataset):
    def __init__(self, rows, processor):
        self.rows = rows
        self.processor = processor

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        pixel_values = self.processor(images=img, return_tensors="pt").pixel_values.squeeze(0)
        labels = self.processor.tokenizer(
            row["label"], padding="max_length", max_length=64,
            truncation=True, return_tensors="pt",
        ).input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels}


def load_rows(quality_filter):
    rows = []
    with LABELS_CSV.open(encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row["quality"] in quality_filter and Path(row["filepath"]).exists():
                rows.append(row)
    return rows


def stratified_split(rows, val_ratio=0.1, seed=42):
    rng = random.Random(seed)
    by_tier = {}
    for row in rows:
        by_tier.setdefault(row["quality"], []).append(row)
    train, val = [], []
    for tier_rows in by_tier.values():
        rng.shuffle(tier_rows)
        n_val = max(1, int(len(tier_rows) * val_ratio))
        val.extend(tier_rows[:n_val])
        train.extend(tier_rows[n_val:])
    rng.shuffle(train); rng.shuffle(val)
    return train, val


def compute_cer(predictions, references):
    total_edits = total_chars = 0
    for pred, ref in zip(predictions, references):
        n, m = len(ref), len(pred)
        dp = list(range(m + 1))
        for i in range(1, n + 1):
            prev, dp[0] = dp[0], i
            for j in range(1, m + 1):
                prev, dp[j] = dp[j], prev if ref[i-1] == pred[j-1] else 1 + min(prev, dp[j], dp[j-1])
        total_edits += dp[m]
        total_chars += max(n, 1)
    return total_edits / total_chars if total_chars else 0.0

print("Dataset helpers loaded.")

In [ ]:
# Cell 6 — Load data + model
rows = load_rows(QUALITY)
train_rows, val_rows = stratified_split(rows)
print(f"Dataset: {len(rows)} total | {len(train_rows)} train | {len(val_rows)} val")

base = MODEL_OUTPUT  # fine-tune from your existing checkpoint
print(f"Loading from {base}...")
processor = TrOCRProcessor.from_pretrained(base)
model = VisionEncoderDecoderModel.from_pretrained(base)
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model.to(device)

train_loader = DataLoader(KillfeedDataset(train_rows, processor), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(KillfeedDataset(val_rows,   processor), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"Batches per epoch: {len(train_loader)}")

In [ ]:
# Cell 7 — Training loop
optimizer = AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps)

best_cer = float("inf")
log_every = max(1, len(train_loader) // 20)
out_path = Path(MODEL_OUTPUT)

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    train_loss = 0.0
    for batch_idx, batch in enumerate(train_loader, 1):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)
        loss = model(pixel_values=pixel_values, labels=labels).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        train_loss += loss.item()
        if batch_idx % log_every == 0 or batch_idx == len(train_loader):
            print(f"  Epoch {epoch}/{EPOCHS} | batch {batch_idx}/{len(train_loader)} | loss {train_loss/batch_idx:.4f}", flush=True)

    train_loss /= len(train_loader)

    # --- Validate ---
    model.eval()
    all_preds, all_refs = [], []
    with torch.no_grad():
        for batch in val_loader:
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)
            generated = model.generate(pixel_values, max_new_tokens=64)
            preds = processor.batch_decode(generated, skip_special_tokens=True)
            label_ids = batch["labels"].clone()
            label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
            refs = processor.batch_decode(label_ids, skip_special_tokens=True)
            all_preds.extend(preds); all_refs.extend(refs)

    cer = compute_cer(all_preds, all_refs)
    print(f"Epoch {epoch:02d}/{EPOCHS} | loss={train_loss:.4f} | CER={cer:.4f}")

    if cer < best_cer:
        best_cer = cer
        tmp = Path(tempfile.mkdtemp(dir=out_path.parent, prefix=".trocr_tmp_"))
        try:
            model.save_pretrained(tmp); processor.save_pretrained(tmp)
            if out_path.exists(): shutil.rmtree(out_path)
            tmp.rename(out_path)
        except Exception:
            shutil.rmtree(tmp, ignore_errors=True); raise
        print(f"  [BEST] CER={best_cer:.4f} — saved to {out_path}")

print(f"\nTraining complete. Best CER: {best_cer:.4f}")
print("[OK] Target achieved!" if best_cer < 0.10 else "[WARN] CER >= 0.10 — consider more data or epochs.")

In [ ]:
# Cell 8 — Copy trained model back to Google Drive
import shutil
shutil.copytree(MODEL_OUTPUT, DRIVE_MODEL_OUT, dirs_exist_ok=True)
print(f"Model saved to Drive: {DRIVE_MODEL_OUT}")